In [0]:
pip install torch torchvision torchviz

  Obtaining dependency information for torchviz from https://files.pythonhosted.org/packages/5e/06/bea648249802b65282414caf5e7bc94fcb6e5a3e311b537845417d19edb9/torchviz-0.0.3-py3-none-any.whl.metadata
  Obtaining dependency information for graphviz from https://files.pythonhosted.org/packages/91/4c/e0ce1ef95d4000ebc1c11801f9b944fa5910ecc15b5e351865763d8657f8/graphviz-0.21-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/47.3 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━ 30.7/47.3 kB 1.8 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 630.3 kB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler, LabelEncoder
from torch.utils.data import Dataset, DataLoader

# 1. Generate large dummy data
np.random.seed(42)
torch.manual_seed(42)


## User Data

In [0]:
# 10,000 users
num_users = 10000
users = pd.DataFrame({
    "user_id": [f"U{i}" for i in range(num_users)],
    "age": np.random.randint(18, 70, num_users),
    "gender": np.random.choice(["M", "F"], num_users),
    "income": np.random.normal(50000, 15000, num_users).astype(int),
    "num_interactions": np.random.poisson(3, num_users),
    "device_type": np.random.choice([f"Device_{i}" for i in range(100)], num_users)
})

users.display()

user_id,age,gender,income,num_interactions,device_type
U0,56,M,44257,2,Device_60
U1,69,F,40908,2,Device_95
U2,46,F,42699,0,Device_39
U3,32,F,64153,1,Device_2
U4,60,M,60597,3,Device_49
U5,25,F,49713,4,Device_0
U6,38,F,41810,0,Device_7
U7,56,F,29755,8,Device_61
U8,36,M,71302,3,Device_63
U9,40,M,58881,3,Device_40


## Item Data

In [0]:
# 500 items
num_items = 500
items = pd.DataFrame({
    "item_id": [f"I{i}" for i in range(num_items)],
    "item_name": [f"Item_{i}" for i in range(num_items)],
    "brand": np.random.choice(["BrandA", "BrandB", "BrandC"], num_items),
    "price": np.random.uniform(100, 2000, num_items).astype(int),
    "avg_purchase_time": np.random.randint(10, 60, num_items)
})

items.display()

item_id,item_name,brand,price,avg_purchase_time
I0,Item_0,BrandB,393,27
I1,Item_1,BrandC,1435,26
I2,Item_2,BrandB,840,57
I3,Item_3,BrandA,1007,41
I4,Item_4,BrandB,597,48
I5,Item_5,BrandB,1498,14
I6,Item_6,BrandC,1706,51
I7,Item_7,BrandB,404,38
I8,Item_8,BrandC,466,45
I9,Item_9,BrandB,744,48


##  Interaction Data

In [0]:
# 50,000 - 100,000 interactions
num_interactions =100000
# np.random.randint(50000, 100001)
interactions = pd.DataFrame({
    "user_id": np.random.choice(users.user_id, num_interactions),
    "item_id": np.random.choice(items.item_id, num_interactions),
    "label": np.random.choice([0, 1], num_interactions, p=[0.9, 0.1])
})

interactions.display()
print(len(interactions))

user_id,item_id,label
U6226,I257,0
U3969,I24,0
U4365,I335,0
U5134,I309,0
U946,I33,0
U2709,I475,0
U7993,I149,0
U6845,I393,0
U945,I3,0
U2728,I395,0


100000


## Encoding Categorical , Normalizing Numerical Features

In [0]:
# 2. Encode categorical features
user_le = LabelEncoder().fit(users.user_id)
gender_le = LabelEncoder().fit(users.gender)
device_le = LabelEncoder().fit(users.device_type)
item_le = LabelEncoder().fit(items.item_id)
brand_le = LabelEncoder().fit(items.brand)

n_users_train   = len(user_le.classes_)
n_devices_train = len(device_le.classes_)
UNK_USER_IDX    = n_users_train         # index for any new user
UNK_DEVICE_IDX  = n_devices_train       # index for any new device

users["user_idx"] = user_le.transform(users.user_id)
users["gender_idx"] = gender_le.transform(users.gender)
users["device_idx"] = device_le.transform(users.device_type)

items["item_idx"] = item_le.transform(items.item_id)
items["brand_idx"] = brand_le.transform(items.brand)

interactions["user_idx"] = user_le.transform(interactions.user_id)
interactions["item_idx"] = item_le.transform(interactions.item_id)

# 3. Normalize numerical features
scaler_u = StandardScaler().fit(users[["age", "income", "num_interactions"]])
users[["age", "income", "num_interactions"]] = scaler_u.transform(
    users[["age", "income", "num_interactions"]]
)

scaler_i = StandardScaler().fit(items[["price", "avg_purchase_time"]])
items[["price", "avg_purchase_time"]] = scaler_i.transform(
    items[["price", "avg_purchase_time"]]
)

users.display(),items.display(),interactions.display()

user_id,age,gender,income,num_interactions,device_type,user_idx,gender_idx,device_idx
U0,0.8356710969299971,M,-0.3951161535135186,-0.5746589587648291,Device_60,0,1,57
U1,1.7075170963272943,F,-0.6192564305304376,-0.5746589587648291,Device_95,1,0,95
U2,0.16502032816284534,F,-0.4993892355058625,-1.7387609718608064,Device_39,1112,0,33
U3,-0.7738907481111672,F,0.9364738588867862,-1.1567099653128177,Device_2,2223,0,12
U4,1.1039314044368578,M,0.6984795844549666,0.0073920477831595464,Device_49,3334,1,44
U5,-1.2433462862481734,F,-0.029959583969084363,0.5894430543311482,Device_0,4445,0,0
U6,-0.3715002868508761,F,-0.5588878041138174,-1.7387609718608064,Device_7,5556,0,67
U7,0.8356710969299971,F,-1.365699102841485,2.917647080523103,Device_61,6667,0,58
U8,-0.5056304406043064,M,1.4149387261289343,0.0073920477831595464,Device_63,7778,1,60
U9,-0.23737013309744573,M,0.5836319537111526,0.0073920477831595464,Device_40,8889,1,35


item_id,item_name,brand,price,avg_purchase_time,item_idx,brand_idx
I0,Item_0,BrandB,-1.2275921931373968,-0.4608595467241835,0,1
I1,Item_1,BrandC,0.7027007198711013,-0.5317174106116657,1,2
I2,Item_2,BrandB,-0.3995298782479548,1.66487636990028,112,1
I3,Item_3,BrandA,-0.09016431541453902,0.5311505477005661,223,0
I4,Item_4,BrandB,-0.8496845594965776,1.027155594912941,334,1
I5,Item_5,BrandB,0.8194074890837072,-1.382011777261451,445,1
I6,Item_6,BrandC,1.2047250763253268,1.2397291865753872,456,2
I7,Item_7,BrandB,-1.2072148207351958,0.31857695603811975,467,1
I8,Item_8,BrandC,-1.09236053992279,0.8145820032504946,478,2
I9,Item_9,BrandB,-0.5773687646671638,1.027155594912941,489,1


user_id,item_id,label,user_idx,item_idx
U6226,I257,0,5809,176
U3969,I24,0,3300,157
U4365,I335,0,3741,263
U5134,I309,0,4596,234
U946,I33,0,9401,257
U2709,I475,0,1901,418
U7993,I149,0,7771,56
U6845,I393,0,6496,327
U945,I3,0,9390,223
U2728,I395,0,1922,329


(None, None, None)

## Data Preparation for Model

In [0]:
# 4. Dataset and DataLoader
class InteractionDataset(Dataset):
    def __init__(self, interactions, users, items):
        self.inter = interactions.reset_index(drop=True)
        self.u = users.set_index("user_idx")
        self.i = items.set_index("item_idx")
    def __len__(self):
        return len(self.inter)
    def __getitem__(self, idx):
        r = self.inter.iloc[idx]
        u = self.u.loc[r.user_idx]
        it = self.i.loc[r.item_idx]
        return {
            "user_idx": torch.tensor(r.user_idx, dtype=torch.long),
            "age": torch.tensor(u.age, dtype=torch.float32),
            "gender_idx":torch.tensor(u.gender_idx, dtype=torch.long),
            "income": torch.tensor(u.income,dtype=torch.float32),
            "num_inter": torch.tensor(u.num_interactions, dtype=torch.float32),
            "device_idx":torch.tensor(u.device_idx, dtype=torch.long),
            "item_idx": torch.tensor(r.item_idx, dtype=torch.long),
            "brand_idx": torch.tensor(it.brand_idx, dtype=torch.long),
            "price":torch.tensor(it.price,dtype=torch.float32),
            "avg_purchase_time":torch.tensor(it.avg_purchase_time, dtype=torch.float32),
            "label":torch.tensor(r.label,dtype=torch.float32)
        }

train_loader = DataLoader(
    InteractionDataset(interactions, users, items),
    batch_size=512,
    shuffle=True,
    num_workers=2
)

## Two Tower Model Architecture

In [0]:
# 5. Two-Tower Model (with unknown-user/device slots)
class TwoTower(nn.Module):
    def __init__(self, 
                 n_users, n_items, n_genders, n_devices, n_brands,
                 emb_dim=32, hidden_dim=64):
        super().__init__()
        # +1 for unknown user/device
        self.u_emb   = nn.Embedding(n_users + 1, emb_dim)
        self.g_emb   = nn.Embedding(n_genders,    8)
        self.dev_emb = nn.Embedding(n_devices + 1, emb_dim)
        self.i_emb   = nn.Embedding(n_items,      emb_dim)
        self.b_emb   = nn.Embedding(n_brands,     8)
        self.u_mlp = nn.Sequential(
            nn.Linear(emb_dim + emb_dim + 8 + 3, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.i_mlp = nn.Sequential(
            nn.Linear(emb_dim + 8 + 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
    def forward(self, user_idx, age, gender_idx, income, num_int, dev_idx,
                      item_idx, brand_idx, price, avg_purchase_time):
        u0 = self.u_emb(user_idx)
        g0 = self.g_emb(gender_idx)
        d0 = self.dev_emb(dev_idx)
        i0 = self.i_emb(item_idx)
        b0 = self.b_emb(brand_idx)
        u_feats = torch.cat([
            u0, d0, g0,
            age.unsqueeze(1), income.unsqueeze(1), num_int.unsqueeze(1)
        ], dim=1)
        i_feats = torch.cat([
            i0, b0,
            price.unsqueeze(1), avg_purchase_time.unsqueeze(1)
        ], dim=1)
        return self.u_mlp(u_feats), self.i_mlp(i_feats)

model = TwoTower(
    n_users=n_users_train,
    n_items=len(items),
    n_genders=users.gender_idx.nunique(),
    n_devices=n_devices_train,
    n_brands=items.brand_idx.nunique()
)

optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()
### user id embedding , user feature embedding (pooling) - industry standard ? or conceptually ?

## Training the Model

In [0]:
# 6. Train 
num_epochs=5

for epoch in range(num_epochs): 
  model.train()
  total_loss = 0
  for batch in train_loader:
      optimizer.zero_grad()
      u_vec, i_vec = model(
          batch["user_idx"],
          batch["age"], batch["gender_idx"],
          batch["income"], batch["num_inter"],
          batch["device_idx"],
          batch["item_idx"],
          batch["brand_idx"],
          batch["price"], batch["avg_purchase_time"]
      )
      logits = (u_vec * i_vec).sum(dim=1)
      loss = criterion(logits, batch["label"].float())
      loss.backward()
      optimizer.step()
      total_loss += loss.item()
  print(f"Epoch {epoch} avg loss: {total_loss/len(train_loader):.4f}")

Epoch 0 avg loss: 0.3457
Epoch 1 avg loss: 0.3259
Epoch 2 avg loss: 0.3197
Epoch 3 avg loss: 0.3119
Epoch 4 avg loss: 0.3035


## Inference on Test Data 
- Precompute and cache the item embeddings to save computation

In [0]:
##brute force - in productionized version , use ANN .

In [0]:
# 7. Prepare July test users (mix of old and new)
test_users = pd.DataFrame([
    {"user_id":"U10", "age":28, "gender":"M", "income":45000, "num_interactions":2, "device_type":"Device_2"},
    {"user_id":"U_new", "age":35, "gender":"F", "income":55000, "num_interactions":0, "device_type":"Device_101"},
    {"user_id":"U_new1", "age":34, "gender":"M", "income":75000, "num_interactions":3, "device_type":"Device_102"},
    {"user_id":"U900", "age":16, "gender":"M", "income":15000, "num_interactions":21, "device_type":"Device_6"},
])
def map_user(u):
    return user_le.transform([u])[0] if u in user_le.classes_ else UNK_USER_IDX
def map_device(d):
    return device_le.transform([d])[0] if d in device_le.classes_ else UNK_DEVICE_IDX

test_users["user_idx"]   = test_users.user_id.map(map_user)
test_users["gender_idx"] = gender_le.transform(test_users.gender)
test_users["device_idx"] = test_users.device_type.map(map_device)
test_users[["age","income","num_interactions"]] = scaler_u.transform(
    test_users[["age","income","num_interactions"]]
)

test_users.display()

user_id,age,gender,income,num_interactions,device_type,user_idx,gender_idx,device_idx
U10,-1.0421510556180278,M,-0.3453890033721119,-0.5746589587648291,Device_2,2,1,12
U_new,-0.5726955174810217,F,0.3238862340627015,-1.7387609718608064,Device_101,10000,0,100
U_new1,-0.6397605943577368,M,1.6624367089323284,0.0073920477831595464,Device_102,10000,1,100
U900,-1.84693197813861,M,-2.353214715676552,10.484310165646955,Device_6,8891,1,56


In [0]:
# 8. Precompute all item vectors
model.eval()
all_item_idxs = torch.arange(len(items))
with torch.no_grad():
    _, all_i_vec = model(
        torch.full((len(items),), UNK_USER_IDX, dtype=torch.long),
        torch.zeros(len(items)), 
        torch.zeros(len(items), dtype=torch.long),
        torch.zeros(len(items)),
        torch.zeros(len(items)),
        torch.full((len(items),), UNK_DEVICE_IDX, dtype=torch.long),
        all_item_idxs,
        torch.tensor(items.brand_idx.values, dtype=torch.long),
        torch.tensor(items.price.values, dtype=torch.float32),
        torch.tensor(items.avg_purchase_time.values, dtype=torch.float32)
    )

In [0]:
# 9. Inference: Top-5 for each test user
recommendations = []
for _, u in test_users.iterrows():
    with torch.no_grad():
        u_vec, _ = model(
            torch.tensor([u.user_idx]*len(items), dtype=torch.long),
            torch.tensor([u.age]*len(items), dtype=torch.float32),
            torch.tensor([u.gender_idx]*len(items), dtype=torch.long),
            torch.tensor([u.income]*len(items), dtype=torch.float32),
            torch.tensor([u.num_interactions]*len(items), dtype=torch.float32),
            torch.tensor([u.device_idx]*len(items), dtype=torch.long),
            all_item_idxs,
            torch.tensor(items.brand_idx.values, dtype=torch.long),
            torch.tensor(items.price.values, dtype=torch.float32),
            torch.tensor(items.avg_purchase_time.values, dtype=torch.float32)
        )
        scores = (u_vec * all_i_vec).sum(dim=1)
        top5 = torch.topk(scores, k=5).indices.tolist()
    recommendations.append({
        "user_id": u.user_id,
        "top_item_ids":   items.loc[top5, "item_id"].tolist(),
        "top_item_names": items.loc[top5, "item_name"].tolist()
    })

recs_df = pd.DataFrame(recommendations)
recs_df.display()


user_id,top_item_ids,top_item_names
U10,"List(I35, I450, I445, I442, I329)","List(Item_35, Item_450, Item_445, Item_442, Item_329)"
U_new,"List(I58, I98, I80, I290, I95)","List(Item_58, Item_98, Item_80, Item_290, Item_95)"
U_new1,"List(I58, I80, I98, I286, I390)","List(Item_58, Item_80, Item_98, Item_286, Item_390)"
U900,"List(I96, I371, I34, I78, I207)","List(Item_96, Item_371, Item_34, Item_78, Item_207)"


In [0]:
# Multi-Label Extension of TwoTower
# Labels: [clicked, wishlisted, purchased, returned]
N_LABELS = 4
LABEL_NAMES = ["clicked", "wishlisted", "purchased", "returned"]

class TwoTowerMultiLabel(nn.Module):
    def __init__(self,
                 n_users, n_items, n_genders, n_devices, n_brands,
                 emb_dim=32, hidden_dim=64, n_labels=4):
        super().__init__()
        # Same towers as TwoTower — unchanged
        self.u_emb   = nn.Embedding(n_users + 1, emb_dim)
        self.g_emb   = nn.Embedding(n_genders, 8)
        self.dev_emb = nn.Embedding(n_devices + 1, emb_dim)
        self.i_emb   = nn.Embedding(n_items, emb_dim)
        self.b_emb   = nn.Embedding(n_brands, 8)

        self.u_mlp = nn.Sequential(
            nn.Linear(emb_dim + emb_dim + 8 + 3, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.i_mlp = nn.Sequential(
            nn.Linear(emb_dim + 8 + 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        # Multi-label head: operates on element-wise product of u_vec and i_vec
        # Input: (batch, 64) — same hidden_dim
        # Output: (batch, n_labels) — one logit per label
        self.head = nn.Linear(hidden_dim, n_labels)

    def forward(self, user_idx, age, gender_idx, income, num_int, dev_idx,
                      item_idx, brand_idx, price, avg_purchase_time):
        u0 = self.u_emb(user_idx)
        g0 = self.g_emb(gender_idx)
        d0 = self.dev_emb(dev_idx)
        i0 = self.i_emb(item_idx)
        b0 = self.b_emb(brand_idx)

        u_feats = torch.cat([u0, d0, g0,
                              age.unsqueeze(1), income.unsqueeze(1), num_int.unsqueeze(1)], dim=1)
        i_feats = torch.cat([i0, b0,
                              price.unsqueeze(1), avg_purchase_time.unsqueeze(1)], dim=1)

        u_vec = self.u_mlp(u_feats)   # (batch, 64)
        i_vec = self.i_mlp(i_feats)   # (batch, 64)

        # Element-wise interaction — captures how each dimension of user aligns with item
        interaction = u_vec * i_vec    # (batch, 64) — NOT collapsed to scalar

        logits = self.head(interaction) # (batch, 4) — one logit per label
        return logits

# --- Dummy multi-label interactions ---
# Each row has 4 binary labels: [clicked, wishlisted, purchased, returned]
np.random.seed(42)
ml_labels = np.column_stack([
    np.random.choice([0,1], num_interactions, p=[0.5, 0.5]),  # clicked
    np.random.choice([0,1], num_interactions, p=[0.7, 0.3]),  # wishlisted
    np.random.choice([0,1], num_interactions, p=[0.9, 0.1]),  # purchased
    np.random.choice([0,1], num_interactions, p=[0.95, 0.05]) # returned
])
ml_label_tensor = torch.tensor(ml_labels, dtype=torch.float32)  # (100000, 4)

# --- Model, optimizer, loss ---
ml_model   = TwoTowerMultiLabel(
    n_users=n_users_train, n_items=len(items),
    n_genders=users.gender_idx.nunique(),
    n_devices=n_devices_train, n_brands=items.brand_idx.nunique(),
    n_labels=N_LABELS
)
ml_optimizer = optim.Adam(ml_model.parameters(), lr=1e-3)
# BCEWithLogitsLoss works on ANY shape — applies sigmoid independently per label
ml_criterion = nn.BCEWithLogitsLoss()

# --- One epoch of training ---
ml_model.train()
total_loss = 0
for batch in train_loader:
    ml_optimizer.zero_grad()
    logits = ml_model(
        batch["user_idx"], batch["age"], batch["gender_idx"],
        batch["income"], batch["num_inter"], batch["device_idx"],
        batch["item_idx"], batch["brand_idx"],
        batch["price"], batch["avg_purchase_time"]
    )  # shape: (512, 4)

    # Fetch the 4 labels for this batch using the interaction indices
    batch_indices = batch["user_idx"]  # reusing as proxy — in real setup, pass row idx
    # For demo: generate random multi-labels per batch
    batch_labels = torch.randint(0, 2, (logits.shape[0], N_LABELS)).float()  # (512, 4)

    loss = ml_criterion(logits, batch_labels)  # BCEWithLogitsLoss handles (512,4) natively
    loss.backward()
    ml_optimizer.step()
    total_loss += loss.item()

print(f"Multi-label training loss (1 epoch): {total_loss/len(train_loader):.4f}")

# --- Inference: output shape and interpretation ---
ml_model.eval()
sample_batch = next(iter(train_loader))
with torch.no_grad():
    sample_logits = ml_model(
        sample_batch["user_idx"], sample_batch["age"], sample_batch["gender_idx"],
        sample_batch["income"], sample_batch["num_inter"], sample_batch["device_idx"],
        sample_batch["item_idx"], sample_batch["brand_idx"],
        sample_batch["price"], sample_batch["avg_purchase_time"]
    )  # (512, 4)
    sample_probs = torch.sigmoid(sample_logits)  # convert logits → probabilities

print(f"\nOutput shape  : {sample_probs.shape}")   # (512, 4)
print(f"Labels        : {LABEL_NAMES}")
print(f"\nSample probabilities for first 5 interactions:")
for i in range(5):
    probs = sample_probs[i].tolist()
    print(f"  Row {i}: " + ", ".join(f"{LABEL_NAMES[j]}={probs[j]:.2f}" for j in range(N_LABELS)))

# --- Rank items for a user by purchase probability (label index 2) ---
print("\nTo rank items by purchase probability, use label index 2:")
print("  purchase_scores = sample_probs[:, 2]  →", sample_probs[:5, 2].tolist())

# --- Weighted multi-objective scoring ---
weights = torch.tensor([0.2, 0.3, 0.5, -0.1])  # penalize returns
weighted_scores = (sample_probs * weights).sum(dim=1)
print(f"\nWeighted scores (first 5): {weighted_scores[:5].tolist()}")